# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fawadwazir/flyrank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 0. Setup — load the starter dataset

Lane 2 (Refresh / Content Opportunity Scoring) works off `data/raw/content_refresh_anonymized.csv` — the 30k-row starter slice, no warehouse/HF token needed for this baseline. Full column reference: `docs/data-dictionary.md`.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows, {df['client_id'].nunique()} clients")
df[["content_id","client_id","impressions_90d","avg_position","position_tier",
    "impression_tier","freshness_tier","days_since_last_update","ctr","clicks_90d"]].head(3)

30,000 rows, 32 clients


,content_id,client_id,impressions_90d,avg_position,position_tier,impression_tier,freshness_tier,days_since_last_update,ctr,clicks_90d
0,content_304f48230142,client_f369cb89fc,3803,10.6,striking,good,0-30,20,0.76,29
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,page_3_5,good,0-30,25,0.05,7
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,page_3_5,good,0-30,20,0.09,11


## 1. My rule and its reason codes

**The rule, in plain words:** *A page is worth a refresh review if it already gets real search visibility, it's gone stale exactly in the window where staleness actually tracks decline (not just "old"), and it currently sits close enough to page 1 that a push could realistically move it — not already winning, not buried too deep to matter.*

**Two signals checked first (before the rule is coded):**

1. **Staleness → behind FlyRank's refresh flags.** Claim: "the longer since a page was last updated, the more likely it's declining." Tested as a bucket table: `freshness_tier` × decline rate (`trend_direction == 'down'`), with `n` printed per bucket.
2. **Position → behind FlyRank's CTR-fix logic.** Claim: "CTR falls as average position gets worse." Tested as a bucket table: `position_tier` × volume-weighted CTR (`sum(clicks_90d)/sum(impressions_90d)`, not the mean of per-page CTRs — averaging per-row rates hides the true rate), with `n` printed per bucket.

Both signals are flag-linked (refresh flags and the CTR-fix logic from this week's session). Verdicts are written after the tables below, in plain words — including if a verdict comes back mixed or false; a clean negative here changes what the rule gates on.


In [2]:
# --- Signal 1: staleness vs decline rate (freshness_tier) ---
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

sig1 = (
    df.groupby("freshness_tier")
      .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
      .round(3)
      .sort_values("decline_rate")
)
print("Signal 1 — freshness_tier vs decline rate (n floor = 50)")
print(sig1)
print()

# --- Signal 2: position tier vs volume-weighted CTR (not mean-of-rates) ---
pos = df[df["position_tier"] != "no_data"].copy()
sig2 = (
    pos.groupby("position_tier")
       .apply(lambda g: pd.Series({
           "n": len(g),
           "weighted_ctr_pct": g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100,
       }))
)
# order by real position quality (best to worst), not alphabetically
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
sig2 = sig2.loc[order].round(4)
print("Signal 2 — position_tier vs volume-weighted CTR% (n floor = 50)")
print(sig2)

Signal 1 — freshness_tier vs decline rate (n floor = 50)
                    n  decline_rate
freshness_tier                     
181+              174         0.471
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611

Signal 2 — position_tier vs volume-weighted CTR% (n floor = 50)
                     n  weighted_ctr_pct
position_tier                           
top_3           2321.0            0.4885
page_1         11814.0            0.3503
striking        7304.0            0.3469
page_3_5        7242.0            0.1549
deep            1319.0            0.0414


**Verdict 1 — staleness vs decline: MIXED.** All four buckets clear the n=50 floor. Decline rate rises from `0-30` (51.1%, n=20,480) through `31-90` (58.9%, n=175) and peaks at `91-180` (61.1%, n=9,171) — that part matches the story FlyRank's refresh flag tells. But it does **not** keep climbing: `181+` (the stalest bucket, n=174) drops back to 47.1% — *lower* than even the freshest pages. A plain "older = more declining" rule would be wrong at the tail. **This is the negative that saves the rule**: staleness only tracks decline in the `91-180` window, not linearly beyond it, so the rule below gates on that specific tier rather than "days_since_last_update ≥ some cutoff."

**Verdict 2 — position vs CTR: CONFIRMED.** Every bucket clears the n=50 floor easily (smallest is `deep` at n=1,319). CTR falls in the expected order as position worsens — `top_3` (0.49%) → `page_1`/`striking` (~0.35%, effectively tied) → `page_3_5` (0.15%) → `deep` (0.04%). The classic CTR-position curve holds in this data, which is what backs FlyRank's CTR-fix logic. It also tells the rule where the real opportunity sits: `page_1`/`striking` pages already get meaningful clicks per impression and have room to gain more if pushed higher; `page_3_5`/`deep` pages are too far down the curve for a refresh alone to move the needle much.


## 2. Build the ranked queue (writes the CSV)

**The rule, coded from the two verdicts above — no fitted weights, just gates and a readable multiply:**

- `stale` = `freshness_tier == '91-180'` (the *only* tier where Verdict 1 actually showed elevated decline — not "old" in general)
- `strikable` = `position_tier` in `{page_1, striking}` (Verdict 2's realistic-opportunity zone — not already `top_3`, not buried in `page_3_5`/`deep`)
- `visible` = `impression_tier` in `{moderate, good, excellent}` (`impressions_90d ≥ 300` — clears the low-volume noise floor the data dictionary warns about for tier reads)

`score = impressions_90d × stale × strikable × visible` — zero unless every gate passes, and among qualifiers, ranked by how much search audience is actually in play. Every column feeding the score is trailing-90-day, already-observed history — **no `trend_direction`, `trend_pct`, or `is_declining_label` anywhere in the gates or the score** (checked explicitly at the end of section 4).


In [3]:
stale       = (df["freshness_tier"] == "91-180").astype(int)
strikable   = df["position_tier"].isin(["page_1", "striking"]).astype(int)
visible     = df["impression_tier"].isin(["moderate", "good", "excellent"]).astype(int)

df["score"]  = df["impressions_90d"] * stale * strikable * visible
df["gates_passed"] = stale + strikable + visible  # 0-3, for the weak-picks look in section 4

df["reason_code"] = np.where(df["score"] > 0, "stale_strikable_visible", "no_flag")

priority_cut = df.loc[df["score"] > 0, "score"].quantile(0.90)
df["action"] = np.select(
    [df["score"] >= priority_cut if priority_cut > 0 else df["score"] > 0,
     df["score"] > 0],
    ["refresh_priority", "refresh_backlog"],
    default="no_action",
)

n_flagged = (df["score"] > 0).sum()
print(f"{n_flagged:,} of {len(df):,} pages pass all three gates ({n_flagged/len(df)*100:.1f}%)")
print(df["action"].value_counts())

4,471 of 30,000 pages pass all three gates (14.9%)
action
no_action           25529
refresh_backlog      4022
refresh_priority      449
Name: count, dtype: int64


In [4]:
queue_cols = [
    "content_id", "client_id", "score", "reason_code", "action",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
    "freshness_tier", "days_since_last_update", "content_type", "main_intent",
    "search_volume", "competition_level",
]
queue = df[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
queue.head(10)

wrote 30,000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action,impressions_90d,clicks_90d,ctr,avg_position,position_tier,freshness_tier,days_since_last_update,content_type,main_intent,search_volume,competition_level
0,content_5fe46e04994d,client_4e07408562,517715,stale_strikable_visible,refresh_priority,517715,741,0.14,4.2,page_1,91-180,104,keyword article,informational,1900.0,LOW
1,content_2c2606c5d176,client_19581e27de,347399,stale_strikable_visible,refresh_priority,347399,1854,0.53,4.2,page_1,91-180,104,keyword article,commercial,590.0,LOW
2,content_cb112fce36be,client_19581e27de,309910,stale_strikable_visible,refresh_priority,309910,492,0.16,5.6,page_1,91-180,104,keyword article,transactional,70.0,MEDIUM
3,content_36ff89c8214e,client_19581e27de,295097,stale_strikable_visible,refresh_priority,295097,154,0.05,7.3,page_1,91-180,104,keyword article,informational,0.0,LOW
4,content_c21024970297,client_19581e27de,211366,stale_strikable_visible,refresh_priority,211366,870,0.41,5.1,page_1,91-180,104,keyword article,commercial,110.0,LOW
5,content_c8e9d6ab9013,client_19581e27de,208678,stale_strikable_visible,refresh_priority,208678,0,0.00,9.7,page_1,91-180,104,keyword article,informational,20.0,LOW
6,content_d17681677e69,client_19581e27de,201584,stale_strikable_visible,refresh_priority,201584,487,0.24,5.8,page_1,91-180,104,keyword article,commercial,10.0,LOW
7,content_a7427266c305,client_19581e27de,201111,stale_strikable_visible,refresh_priority,201111,219,0.11,5.7,page_1,91-180,104,keyword article,informational,0.0,LOW
8,content_c5063073d048,client_6208ef0f77,192205,stale_strikable_visible,refresh_priority,192205,466,0.24,12.5,striking,91-180,104,keyword article,informational,0.0,LOW
9,content_3d94572c3a35,client_19581e27de,190623,stale_strikable_visible,refresh_priority,190623,462,0.24,4.3,page_1,91-180,104,keyword article,commercial,50.0,LOW


## 3. Top-10 review

For each of the top 10 by score: the action, why it's there (pulled from its own row), and what would make it wrong — read with a skeptic's eye, not a rubber stamp.


In [5]:
top10 = queue.head(10).reset_index(drop=True)

for i, r in top10.iterrows():
    print(f"#{i+1}  {r['content_id']}  (client {r['client_id']})")
    print(f"   action: {r['action']}   score: {r['score']:,.0f}   reason_code: {r['reason_code']}")
    print(f"   why: impressions_90d={r['impressions_90d']:,.0f} (visible), "
          f"freshness_tier={r['freshness_tier']} (stale window), "
          f"avg_position={r['avg_position']:.1f} / {r['position_tier']} (strikable)")

    flags = []
    if r["competition_level"] == "HIGH":
        flags.append("HIGH keyword competition — a refresh may not be enough to move rank")
    if pd.notna(r["search_volume"]) and r["search_volume"] < 50:
        flags.append(f"underlying keyword volume is only {r['search_volume']:.0f} — impressions may come mostly from long-tail queries a refresh won't touch")
    if r["content_type"] == "feedly article":
        flags.append("feedly (news-style) content — often time-bound; a refresh may not extend its shelf life the way it would for evergreen content")
    if r["main_intent"] in ("navigational",):
        flags.append("navigational intent — clicks may be brand-driven, not content-quality-driven, so a refresh may not lift CTR")
    if not flags:
        flags.append("no obvious red flag in these columns — the main risk is the usual one: the update timestamp is stale in our data but the page may already have been touched off-cycle")
    print(f"   what would make it wrong: {'; '.join(flags)}")
    print()

#1  content_5fe46e04994d  (client client_4e07408562)
   action: refresh_priority   score: 517,715   reason_code: stale_strikable_visible
   why: impressions_90d=517,715 (visible), freshness_tier=91-180 (stale window), avg_position=4.2 / page_1 (strikable)
   what would make it wrong: no obvious red flag in these columns — the main risk is the usual one: the update timestamp is stale in our data but the page may already have been touched off-cycle

#2  content_2c2606c5d176  (client client_19581e27de)
   action: refresh_priority   score: 347,399   reason_code: stale_strikable_visible
   why: impressions_90d=347,399 (visible), freshness_tier=91-180 (stale window), avg_position=4.2 / page_1 (strikable)
   what would make it wrong: no obvious red flag in these columns — the main risk is the usual one: the update timestamp is stale in our data but the page may already have been touched off-cycle

#3  content_cb112fce36be  (client client_19581e27de)
   action: refresh_priority   score: 309,91

## 4. Weak picks + leakage check

**Weak picks — found by actually running this, not guessed in advance:**

- **5 of the top 10** show `search_volume` of 0–20 against impressions in the hundreds of thousands. That's not a coding error — it means the page ranks for a much broader set of queries than its one tracked "target keyword," so `search_volume` badly understates the real opportunity for these rows. The score doesn't use `search_volume` at all, so it can't be gated on directly; it only shows up as a caution in the top-10 review.
- **8 of the top 10 belong to a single client** (`client_19581e27de`). The rule has no per-client fairness or cap built in — one client with unusually large raw traffic can dominate the entire ranked queue, effectively burying every smaller client's pages no matter how strong their individual case is. A real deployment would need either a per-client quota or a within-client percentile score instead of a raw cross-client one.

Neither weakness shows up in the score itself — both are exactly the kind of miss a three-input rule is expected to make, which is the point of reading the top of the list by hand instead of trusting the ranking blind.

**Leakage check** — confirm none of the label-derived or future-window columns leaked into the gates or the score:


In [6]:
forbidden = {"trend_direction", "trend_pct", "is_declining_label", "is_declining",
             "impressions_last_30d", "impressions_prev_30d"}
score_inputs = {"freshness_tier", "position_tier", "impression_tier", "impressions_90d"}

leaked = forbidden & score_inputs
print("forbidden columns used in the score/gates:", leaked if leaked else "none")
assert not leaked, "leakage detected — a label-derived or future-window column reached the score"
print("clean: score is built only from trailing-90d, already-observed columns.")

forbidden columns used in the score/gates: none
clean: score is built only from trailing-90d, already-observed columns.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
